In [18]:
#25.Train a GradientBoostingClassifier inside your existing pipeline and evaluate it with cross validation.
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import time
titanic = sb.load_dataset("titanic")
X = titanic.drop(["survived", "alive"], axis=1)
y = titanic["survived"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X.select_dtypes(include=["object", "string", "category", "bool"]).columns
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numerical_columns),("categorical", categorical_pipeline, categorical_columns)])
model_pipeline=Pipeline([("preprocessor", preprocessor),("classifier", GradientBoostingClassifier(random_state=42))])
start_time = time.time()
cv_scores=cross_val_score(model_pipeline,X_train,y_train,cv=5,scoring="accuracy")
gb_time = time.time() - start_time
print("CV Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())
print("Gradient Boosting Time:", gb_time, "seconds")

CV Scores: [0.78321678 0.76223776 0.85915493 0.84507042 0.84507042]
Mean CV Score: 0.8189500640204864
Gradient Boosting Time: 1.550422191619873 seconds


In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier
model_pipeline_hist=Pipeline([("preprocessor", preprocessor),("classifier", HistGradientBoostingClassifier(random_state=42))])
start_time = time.time()
cv_scores_hist=cross_val_score(model_pipeline,X_train,y_train,cv=5,scoring="accuracy")
hist_gb_time = time.time() - start_time
print("CV Scores:", cv_scores_hist)
print("Mean CV Score:", cv_scores_hist.mean())
print("HistGradient Boosting Time:", hist_gb_time, "seconds")

CV Scores: [0.78321678 0.76223776 0.85915493 0.84507042 0.84507042]
Mean CV Score: 0.8189500640204864
HistGradient Boosting Time: 1.7196540832519531 seconds


In [20]:
#27.Try three learning rates, one small, one moderate, and one large,
#and watch what happens to training accuracy and validation accuracy separately.
from sklearn.metrics import accuracy_score
learning_rates = [0.01, 0.1, 0.5]
X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train,y_train,test_size=0.2,random_state=42,stratify=y_train)
for rate in learning_rates:
    model=Pipeline([("preprocessor",preprocessor),("classifier",GradientBoostingClassifier(learning_rate=rate,random_state=42))])
    model.fit(X_train_sub,y_train_sub)
    train_predictions=model.predict(X_train_sub)
    val_predictions=model.predict(X_val)
    train_accuracy = accuracy_score(y_train_sub, train_predictions)
    val_accuracy = accuracy_score(y_val, val_predictions)
    print("Learning Rate:", rate)
    print("Training Accuracy:", train_accuracy)
    print("Validation Accuracy:", val_accuracy)
    print()

Learning Rate: 0.01
Training Accuracy: 0.8418277680140598
Validation Accuracy: 0.8391608391608392

Learning Rate: 0.1
Training Accuracy: 0.9121265377855887
Validation Accuracy: 0.8181818181818182

Learning Rate: 0.5
Training Accuracy: 0.9789103690685413
Validation Accuracy: 0.7902097902097902



In [21]:
#28.Turn on early stopping for the histogram version and note how many trees it actually
#chose to build before stopping.
hist_model_new=Pipeline([ ("preprocessor", preprocessor),("classifier", HistGradientBoostingClassifier(max_iter=500,
             early_stopping=True,random_state=42))])
hist_model_new.fit(X_train,y_train)
iterations_used = hist_model_new.named_steps["classifier"].n_iter_
print("Maximum Iterations:", 500)
print("Iterations Actually Used:", iterations_used)

Maximum Iterations: 500
Iterations Actually Used: 47


In [22]:
#29.Produce a full evaluation of your best boosted model using a confusion matrix, precision,
#recall, F1 score, and ROC AUC, exactly as you did on Day 10.
from sklearn.metrics import (confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score)
y_pred = hist_model_new.predict(X_test)
y_prob = hist_model_new.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[99 11]
 [18 51]]
Precision: 0.8225806451612904
Recall: 0.7391304347826086
F1 Score: 0.7786259541984732
ROC AUC: 0.841304347826087
